# Làm sạch dữ liệu

Dữ liệu [../data/house_prices_raw.csv](../data/house_prices_raw.csv) mô phỏng dữ liệu giao dịch nhà thu thập từ thực tế, có đủ các lỗi thường gặp: dòng trùng lặp, chữ viết không thống nhất, giá trị thiếu, giá trị ngoại lai.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path('..') / 'data'                        # thư mục dữ liệu của bài
df = pd.read_csv(DATA / 'house_prices_raw.csv')
print('Kích thước:', df.shape)
df.head()

Kích thước: (1020, 7)


,area,bedrooms,distance,district,year_built,paint_color,price
0,173.2,4.0,9.1,suburb,2024.0,gray,5786.0
1,123.5,4.0,8.3,suburb,2011.0,blue,4675.0
2,129.8,3.0,4.6,center,NaN,white,4984.0
3,69.2,1.0,6.2,suburb,1995.0,blue,1902.0
4,110.6,3.0,15.9,outskirts,2011.0,yellow,3222.0


## 1. Kiểm tra tổng quan

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   area         1020 non-null   float64
 1   bedrooms     1005 non-null   float64
 2   distance     1020 non-null   float64
 3   district     1020 non-null   str    
 4   year_built   979 non-null    float64
 5   paint_color  1020 non-null   str    
 6   price        1014 non-null   float64
dtypes: float64(5), str(2)
memory usage: 67.8 KB


In [3]:
print('Số giá trị thiếu của từng cột:')
print(df.isna().sum())
print('Số dòng trùng lặp:', df.duplicated().sum())

Số giá trị thiếu của từng cột:
area            0
bedrooms       15
distance        0
district        0
year_built     41
paint_color     0
price           6
dtype: int64
Số dòng trùng lặp: 20


In [4]:
df.describe().round(1)

,area,bedrooms,distance,year_built,price
count,1020.0,1005.0,1020.0,979.0,1014.0
mean,127.0,2.9,12.5,2007.2,3818.8
std,117.6,1.4,7.0,10.6,1694.3
min,30.2,1.0,0.5,1990.0,300.0
25%,78.3,2.0,6.7,1998.0,2455.2
50%,121.0,3.0,12.1,2007.0,3906.0
75%,161.6,4.0,18.4,2016.0,5241.2
max,1989.0,6.0,25.0,2025.0,7678.0


**Nhận xét:** diện tích lớn nhất gần 2000 m² trong khi phân vị 75% chỉ khoảng 160 m²: có giá trị **ngoại lai**. Một số cột có giá trị thiếu, và có 20 dòng trùng lặp.

## 2. Xóa dòng trùng lặp

In [5]:
df = df.drop_duplicates()
print('Còn lại:', len(df), 'dòng')

Còn lại: 1000 dòng


## 3. Chuẩn hóa chữ viết trong cột phân loại

In [6]:
print('Trước:', df['district'].unique())
df['district'] = df['district'].str.strip().str.lower()   # bỏ khoảng trắng thừa, đưa về chữ thường
print('Sau  :', df['district'].unique())
df['district'].value_counts()

Trước: <ArrowStringArray>
[    'suburb',     'center',  'outskirts',     'SUBURB',    'Suburb ',
 'Outskirts ', ' outskirts',    ' suburb',     'CENTER',  'OUTSKIRTS',
    ' center',    'Center ']
Length: 12, dtype: str
Sau  : <ArrowStringArray>
['suburb', 'center', 'outskirts']
Length: 3, dtype: str


district
suburb       421
outskirts    394
center       185
Name: count, dtype: int64

## 4. Xử lý giá trị thiếu

- Thiếu **nhãn** (`price`): không thể dùng để huấn luyện, **xóa** các dòng này.
- Thiếu **đặc trưng** (`bedrooms`, `year_built`): số lượng ít, có thể xóa; nhưng để giữ dữ liệu, thường **điền** bằng trung vị (ít bị ảnh hưởng bởi ngoại lai hơn trung bình).

In [7]:
df = df.dropna(subset=['price'])                 # xóa dòng thiếu nhãn
print('Còn lại:', len(df), 'dòng')

df_filled = df.copy()
for column in ['bedrooms', 'year_built']:
    median = df_filled[column].median()          # trung vị của cột
    df_filled[column] = df_filled[column].fillna(median)
    print(f'Điền {column} bằng trung vị {median}')
print(df_filled.isna().sum().sum(), 'giá trị thiếu còn lại')

Còn lại: 995 dòng
Điền bedrooms bằng trung vị 3.0
Điền year_built bằng trung vị 2007.0
0 giá trị thiếu còn lại


Trong một dự án thực tế, trung vị phải được tính trên **tập huấn luyện** rồi mới dùng để điền cho tập kiểm tra (xem dự án ở notebook 08, dùng `SimpleImputer` trong `Pipeline`). Ở đây điền trên toàn bộ dữ liệu chỉ để minh họa.

## 5. Phát hiện và xử lý ngoại lai bằng quy tắc IQR

Với $Q_1, Q_3$ là phân vị 25% và 75%, $\text{IQR} = Q_3 - Q_1$, điểm nằm ngoài khoảng $[Q_1 - 1.5\,\text{IQR},\ Q_3 + 1.5\,\text{IQR}]$ được coi là ngoại lai.

In [8]:
q1, q3 = df['area'].quantile([0.25, 0.75])     # phân vị 25% và 75%
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1 = {q1:.1f}, Q3 = {q3:.1f}, khoảng hợp lệ [{lower:.1f}, {upper:.1f}]')
outliers = df[(df['area'] < lower) | (df['area'] > upper)]
print('Số ngoại lai:', len(outliers))
outliers[['area', 'bedrooms', 'price']]

Q1 = 78.3, Q3 = 161.6, khoảng hợp lệ [-46.6, 286.5]
Số ngoại lai: 6


,area,bedrooms,price
78,1989.0,5.0,6882.0
147,848.0,1.0,2539.0
258,844.0,2.0,3358.0
675,890.0,2.0,2306.0
775,1305.0,3.0,3891.0
842,1427.0,3.0,4926.0


Các căn "1305 m² mà chỉ 3 phòng, giá chưa tới 4 tỷ" rõ ràng là **lỗi nhập liệu** (gõ thừa một chữ số 0), không phải nhà thật sự lớn. Ta xóa các dòng này. Lưu ý: không phải ngoại lai nào cũng là lỗi; trước khi xóa cần kiểm tra bằng hiểu biết về bài toán.

In [9]:
df_clean = df[(df['area'] >= lower) & (df['area'] <= upper)]
print('Dữ liệu sau làm sạch:', df_clean.shape)
df_clean.describe().round(1)

Dữ liệu sau làm sạch: (989, 7)


,area,bedrooms,distance,year_built,price
count,989.0,974.0,989.0,949.0,989.0
mean,118.8,2.9,12.5,2007.1,3816.7
std,48.4,1.4,7.0,10.6,1695.5
min,30.2,1.0,0.5,1990.0,300.0
25%,78.2,2.0,6.6,1998.0,2447.0
50%,120.5,3.0,12.1,2007.0,3912.0
75%,160.3,4.0,18.5,2016.0,5239.0
max,200.0,6.0,25.0,2025.0,7678.0
